In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("../dataset/Fraud detection Dataset.csv")
df.head()

,transaction_id,user_id,transaction_amount,transaction_type,transaction_time,location,is_foreign_transaction,device_type,is_new_device,is_new_location,account_age_days,avg_transaction_amount,transactions_last_24h,failed_logins_24h,time_since_last_txn,txn_hour,is_weekend,is_fraud
0,T0086548,U9625,5068.33,NEFT,25-09-2024 02:47,Delhi,0,Tablet,0,1,3124,4957.53,1,0,947,2,0,0
1,T0088155,U9783,3428.09,IMPS,23-06-2024 07:32,Delhi,0,Mobile,0,1,2302,3750.88,0,0,902,7,1,0
2,T0076282,U8629,5161.81,NEFT,02-06-2024 16:01,Mumbai,0,Mobile,0,0,1157,4075.63,5,0,893,16,1,0
3,T0013138,U2319,4242.40,UPI,30-01-2024 01:36,Chennai,0,Tablet,0,0,613,3745.28,2,1,1139,1,0,0
4,T0060581,U7056,1721.96,UPI,02-07-2024 06:52,Hyderabad,0,Desktop,0,0,637,2818.78,1,0,710,6,0,0


In [5]:
# since the number of unique values in transaction_id and the number of columns present in the dataset are the same
# this implies there are unique transaction id for all the transactions 
# this means there is no predictive power in this column and hence can be removed
df.drop(["transaction_id"], axis = 1, inplace = True)

In [6]:
#since every user id starts with u, we remove this and keep it ready for one hot encoding
df["user_id"] = df["user_id"].str[1:]
df.head()

,user_id,transaction_amount,transaction_type,transaction_time,location,is_foreign_transaction,device_type,is_new_device,is_new_location,account_age_days,avg_transaction_amount,transactions_last_24h,failed_logins_24h,time_since_last_txn,txn_hour,is_weekend,is_fraud
0,9625,5068.33,NEFT,25-09-2024 02:47,Delhi,0,Tablet,0,1,3124,4957.53,1,0,947,2,0,0
1,9783,3428.09,IMPS,23-06-2024 07:32,Delhi,0,Mobile,0,1,2302,3750.88,0,0,902,7,1,0
2,8629,5161.81,NEFT,02-06-2024 16:01,Mumbai,0,Mobile,0,0,1157,4075.63,5,0,893,16,1,0
3,2319,4242.40,UPI,30-01-2024 01:36,Chennai,0,Tablet,0,0,613,3745.28,2,1,1139,1,0,0
4,7056,1721.96,UPI,02-07-2024 06:52,Hyderabad,0,Desktop,0,0,637,2818.78,1,0,710,6,0,0


In [7]:
# transaction amount can be scaled
#transaction type must be one hot encoded
# location must also be one hot encoded
# device is also one hot encoded
# others can be scaled if it involves categorical number

In [8]:
df["transaction_time"] = pd.to_datetime(df["transaction_time"], dayfirst=True)

In [9]:
df["time"] = df["transaction_time"].dt.time
df["date"] = df["transaction_time"].dt.date

In [10]:
df.drop(["transaction_time"], axis = 1, inplace = True)

In [11]:
df["account_age_yrs"] = (df["account_age_days"]/ 365).astype(int)
df["account_age_yrs"].unique()

array([8, 6, 3, 1, 7, 0, 2, 9, 5, 4])

In [12]:
df["is_fraud"].value_counts()
#dataset is severly unbalanced - splitting must be done in such a way that the training must have approx equal number of 1s and 0s

is_fraud
0    90168
1    10018
Name: count, dtype: int64

In [13]:
df.drop(["account_age_days"], axis = 1, inplace = True)
df.head()

,user_id,transaction_amount,transaction_type,location,is_foreign_transaction,device_type,is_new_device,is_new_location,avg_transaction_amount,transactions_last_24h,failed_logins_24h,time_since_last_txn,txn_hour,is_weekend,is_fraud,time,date,account_age_yrs
0,9625,5068.33,NEFT,Delhi,0,Tablet,0,1,4957.53,1,0,947,2,0,0,02:47:00,2024-09-25,8
1,9783,3428.09,IMPS,Delhi,0,Mobile,0,1,3750.88,0,0,902,7,1,0,07:32:00,2024-06-23,6
2,8629,5161.81,NEFT,Mumbai,0,Mobile,0,0,4075.63,5,0,893,16,1,0,16:01:00,2024-06-02,3
3,2319,4242.40,UPI,Chennai,0,Tablet,0,0,3745.28,2,1,1139,1,0,0,01:36:00,2024-01-30,1
4,7056,1721.96,UPI,Hyderabad,0,Desktop,0,0,2818.78,1,0,710,6,0,0,06:52:00,2024-07-02,1


In [14]:
df["date"] = pd.to_datetime(df["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100186 entries, 0 to 100185
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   user_id                 100186 non-null  object        
 1   transaction_amount      100186 non-null  float64       
 2   transaction_type        100186 non-null  object        
 3   location                100186 non-null  object        
 4   is_foreign_transaction  100186 non-null  int64         
 5   device_type             100186 non-null  object        
 6   is_new_device           100186 non-null  int64         
 7   is_new_location         100186 non-null  int64         
 8   avg_transaction_amount  100186 non-null  float64       
 9   transactions_last_24h   100186 non-null  int64         
 10  failed_logins_24h       100186 non-null  int64         
 11  time_since_last_txn     100186 non-null  int64         
 12  txn_hour                100186

In [15]:
from sklearn.model_selection import train_test_split

x = df.drop(["is_fraud"], axis = 1)
y = df["is_fraud"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 42)
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=0.5, random_state = 42)

In [16]:
print(len(y_train[y_train == 1])/ len(x_train))
print(len(y_train[y_train == 0])/ len(x_train))

0.09905889063168402
0.900941109368316


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer([
    ("trf1", StandardScaler(), ["transaction_amount", "avg_transaction_amount","time_since_last_txn"]),
    ("trf2", OneHotEncoder(drop = "first", sparse_output = False, handle_unknown="ignore"), ["user_id", "transaction_type", "location", "device_type", "txn_hour", "time", "date", "account_age_yrs"]),
    
], remainder = "passthrough")

In [18]:
x_train = transformer.fit_transform(x_train)
x_test = transformer.transform(x_test)

c:\Users\Nadakishore\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [19]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state = 42)
x_train, y_train = sm.fit_resample(x_train, y_train)

In [20]:
print(len(y_train[y_train == 1])/ len(x_train))
print(len(y_train[y_train == 0])/ len(x_train))

0.5
0.5


In [21]:
len(x_train)

126366

In [22]:
print(len(y_test[y_test == 1])/ len(x_test))
print(len(y_test[y_test == 0])/ len(x_test))

0.10340697364918819
0.8965930263508118


In [23]:
from sklearn.ensemble import RandomForestClassifier

model1 = RandomForestClassifier(max_depth = 10, n_jobs = -1)
model1.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [24]:
pred = model1.predict_proba(x_test)[:, 1]
pred = (pred >= 0.494).astype(int)

In [25]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

print(accuracy_score(y_test, pred))
print(f1_score(y_test, pred))
print(precision_score(y_test, pred))
print(recall_score(y_test, pred))
print(confusion_matrix(y_test,pred))

0.9284668618578653
0.6835443037974683
0.6299511665762344
0.747104247104247
[[12792   682]
 [  393  1161]]


In [26]:
x_val = transformer.transform(x_val)

c:\Users\Nadakishore\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [27]:
predict = model1.predict_proba(x_val)[:, 1]
pred = (predict >= 0.494).astype(int)

In [28]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

print(accuracy_score(y_val, pred))
print(f1_score(y_val, pred))
print(precision_score(y_val, pred))
print(recall_score(y_val, pred))
print(confusion_matrix(y_val,pred))

0.9293319137609795
0.6795413397706699
0.6265998887033946
0.7422544495715228
[[12840   671]
 [  391  1126]]


In [29]:
import pickle

file = open("model.pkl", "wb")
pickle.dump(model1, file)
file.close()

In [30]:
file = open("preprocessing.pkl", "wb")
pickle.dump(transformer, file)
file.close()

In [31]:
import xgboost as xgb 

model2 = xgb.XGBClassifier()
model2.fit(x_train, y_train)
pred = model2.predict(x_test)

In [33]:
predict = model2.predict_proba(x_test)[:, 1]
pred = (predict >= 0.494).astype(int)

In [34]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

print(accuracy_score(y_test, pred))
print(f1_score(y_test, pred))
print(precision_score(y_test, pred))
print(recall_score(y_test, pred))
print(confusion_matrix(y_test,pred))

0.9752462070801171
0.8664752333094041
0.9797077922077922
0.7767052767052767
[[13449    25]
 [  347  1207]]


In [35]:
predict = model2.predict_proba(x_val)[:, 1]
pred = (predict >= 0.494).astype(int)

In [36]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

print(accuracy_score(y_val, pred))
print(f1_score(y_val, pred))
print(precision_score(y_val, pred))
print(recall_score(y_val, pred))
print(confusion_matrix(y_val,pred))

0.9757785467128027
0.8667642752562226
0.974485596707819
0.7804878048780488
[[13480    31]
 [  333  1184]]


In [37]:
import pickle

file = open("model.pkl", "wb")
pickle.dump(model2, file)
file.close()